# WasteWise VN — Final YOLO26s training (Kaggle GPU only)

Use this notebook with the Kaggle dataset built from `datasets/multiclass_final_v7`. It runs the explicitly limited, manifest-pinned fast preflight, CUDA/DDP preflight, and a short one-epoch smoke before the 50-epoch run. The fast preflight is not equivalent to the committed full local image/hash/near-duplicate audit. It never evaluates the sealed test split.

**Before Run All:** attach the prepared dataset as a Kaggle input; enable GPU and Internet (the reviewed pretrained `yolo26s.pt` must be downloadable if it is not cached). `REVIEWED_COMMIT` below pins the tested Kaggle training source. To persist output, wait for completion and use **Save Version** with notebook output included. `/kaggle/working` is temporary until Kaggle creates a saved notebook version.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = 'https://github.com/ducnguyenhuu26/Project1.git'
REVIEWED_COMMIT = '226ccf2cd5676ef6583ccfe798c6e22e2053e67b'
REPO = Path('/kaggle/working/Project1')
assert len(REVIEWED_COMMIT) == 40 and all(c in '0123456789abcdef' for c in REVIEWED_COMMIT), 'REVIEWED_COMMIT must be the full reviewed Git SHA.'
if not REPO.exists(): subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
subprocess.run(['git','fetch','origin'],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach',REVIEWED_COMMIT],cwd=REPO,check=True)
actual = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
assert actual == REVIEWED_COMMIT, (actual, REVIEWED_COMMIT)
print('Reviewed Git commit:', actual)
os.chdir(REPO)
os.environ['WASTEWISE_EXPECTED_COMMIT'] = REVIEWED_COMMIT

In [ ]:
# Keep Kaggle's CUDA-enabled torch. Pin only Ultralytics if needed; --no-deps avoids replacing torch.
import importlib.metadata as md
try:
    version = md.version('ultralytics')
except md.PackageNotFoundError:
    version = ''
if version != '8.4.210':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'ultralytics==8.4.210'], check=True)
import torch, ultralytics
print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'available', torch.cuda.is_available())
print('ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'Enable Kaggle GPU accelerator before proceeding.'
assert ultralytics.__version__ == '8.4.210'

## Resume after a saved Kaggle version

Attach the prior saved notebook output as an input and set `WASTEWISE_RESUME_RUN` in the next cell to the path containing `run_config.json` and `weights/last.pt`. The runner requires the identical dataset manifest SHA, taxonomy version, class order, pretrained checkpoint SHA, optimizer settings, epoch schedule, and GPU device configuration. It stops if any differ. An unsaved interrupted session cannot be recovered from Kaggle's temporary working storage.

In [ ]:
# Optional resume path, e.g. /kaggle/input/wastewise-previous-output/wastewise_runs/wastewise_yolo26s_v7_50ep
# Set this only when attaching a saved, complete notebook output containing last.pt and run_config.json.
# os.environ['WASTEWISE_RESUME_RUN'] = '/kaggle/input/.../wastewise_yolo26s_v7_50ep'
# Explicitly set to '1' only if a two-GPU DDP smoke fails and you want a fresh one-GPU smoke before training.
# os.environ['WASTEWISE_ALLOW_SINGLE_GPU_FALLBACK'] = '1'
subprocess.run([sys.executable, 'ml/kaggle/final_train.py'], cwd=REPO, check=True)

## Artifacts and deployment

The runner writes `wastewise_artifacts/wastewise_yolo26s_v7_artifacts.zip`, `SHA256SUMS.txt`, model manifests, per-class validation metrics, confusion/PR plots, and epoch checkpoints under `/kaggle/working`. Save a Kaggle notebook version after the run to persist downloadable output. Verify the ZIP checksum before deployment, then use `scripts/install_model.py` with `best.pt` and its adjacent `model_manifest.json`. Keep the prior model directory and `.env` values for rollback. The test split remains sealed until model selection is frozen.